# Twitter Sentiment Analysis: LSTM vs GRU
Run this in Google Colab with a GPU runtime: **Runtime -> Change runtime type -> T4 GPU**

Upload your `twitter_training.csv` when prompted in the cell below (columns: id, entity, sentiment, text — no header).

In [ ]:
!pip install -q tensorflow scikit-learn pandas matplotlib

In [ ]:
# Upload the dataset (skip this cell if the file is already in your Colab environment,
# e.g. mounted from Google Drive, and just set DATA_PATH below instead)
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("Using file:", DATA_PATH)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, SpatialDropout1D, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load data

In [ ]:
df = pd.read_csv(DATA_PATH, header=None,
                  names=["id", "entity", "sentiment", "text"])

df = df.dropna(subset=["text"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["text", "sentiment"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
print(df["sentiment"].value_counts())

## 2. Text cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)          # URLs
    text = re.sub(r"@\w+", " ", text)                     # mentions
    text = re.sub(r"#(\w+)", r"\1", text)                 # hashtags -> keep word
    text = re.sub(r"[^a-z\s']", " ", text)                 # keep letters/apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

## 3. Encode labels

In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentiment"])
num_classes = len(le.classes_)
print("Classes:", list(le.classes_))

X = df["clean_text"].values
y = to_categorical(df["label"].values, num_classes=num_classes)

## 4. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=df["label"].values
)

## 5. Tokenization + padding

In [ ]:
VOCAB_SIZE = 20000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

## 6. Model builders

In [ ]:
EMBED_DIM = 128

def build_lstm_model():
    model = Sequential([
        Input(shape=(MAX_LEN,)),
        Embedding(VOCAB_SIZE, EMBED_DIM),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True)),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(64, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

def build_gru_model():
    model = Sequential([
        Input(shape=(MAX_LEN,)),
        Embedding(VOCAB_SIZE, EMBED_DIM),
        SpatialDropout1D(0.3),
        GRU(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=True),
        GRU(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(64, activation="relu"),
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

## 7. Train LSTM

In [ ]:
print("\n===== Training LSTM =====")
lstm_model = build_lstm_model()
lstm_model.summary()

early_stop = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)

lstm_history = lstm_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=12,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1,
)

lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"LSTM Test Accuracy: {lstm_test_acc*100:.2f}%")

## 8. Train GRU

In [ ]:
print("\n===== Training GRU =====")
gru_model = build_gru_model()
gru_model.summary()

early_stop_gru = EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)

gru_history = gru_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=12,
    batch_size=128,
    callbacks=[early_stop_gru],
    verbose=1,
)

gru_test_loss, gru_test_acc = gru_model.evaluate(X_test_pad, y_test, verbose=0)
print(f"GRU Test Accuracy: {gru_test_acc*100:.2f}%")

## 9. Classification reports

In [ ]:
def report(model, name):
    preds = np.argmax(model.predict(X_test_pad, verbose=0), axis=1)
    true = np.argmax(y_test, axis=1)
    print(f"\n--- {name} Classification Report ---")
    print(classification_report(true, preds, target_names=le.classes_))
    return accuracy_score(true, preds)

lstm_acc = report(lstm_model, "LSTM")
gru_acc = report(gru_model, "GRU")

## 10. Accuracy comparison graphs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) LSTM train vs val accuracy over epochs
axes[0].plot(lstm_history.history["accuracy"], label="Train Acc")
axes[0].plot(lstm_history.history["val_accuracy"], label="Val Acc")
axes[0].set_title("LSTM Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

# (b) GRU train vs val accuracy over epochs
axes[1].plot(gru_history.history["accuracy"], label="Train Acc")
axes[1].plot(gru_history.history["val_accuracy"], label="Val Acc")
axes[1].set_title("GRU Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

# (c) Final test accuracy bar comparison
models_names = ["LSTM", "GRU"]
test_accs = [lstm_test_acc * 100, gru_test_acc * 100]
bars = axes[2].bar(models_names, test_accs, color=["#4C72B0", "#DD8452"])
axes[2].set_title("Final Test Accuracy Comparison")
axes[2].set_ylabel("Accuracy (%)")
axes[2].set_ylim(0, 100)
axes[2].axhline(90, color="green", linestyle="--", linewidth=1, label="90% target (LSTM)")
axes[2].axhline(80, color="red", linestyle="--", linewidth=1, label="80% target (GRU)")
for bar, acc in zip(bars, test_accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, acc + 1, f"{acc:.2f}%",
                 ha="center", fontweight="bold")
axes[2].legend()

plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

print("\n============================================")
print(f"FINAL RESULTS  ->  LSTM: {lstm_acc*100:.2f}%   GRU: {gru_acc*100:.2f}%")
print("============================================")